In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown

BASE = Path.cwd()
if not (BASE / "data" / "market_neutral_prices.csv").exists():
    BASE = Path("final_deliverables")

DATA_FILE = BASE / "data" / "market_neutral_prices.csv"
ASSET_DIR = BASE / "assets"
ASSET_EN_DIR = BASE / "assets_en"

def show_svg(path, width=980):
    """Display an SVG at notebook-friendly size so it fits without scrollbars."""
    svg = Path(path).read_text(encoding="utf-8")
    svg = svg.replace(
        "<svg ",
        f"<svg style='width:100%; max-width:{width}px; height:auto; display:block;' ",
        1
    )
    return HTML(f"<div style='width:100%; max-width:{width}px; overflow:hidden'>{svg}</div>")

START, END = "2013-01-01", "2025-12-31"
LOOKBACK_MONTHS = 6
SKIP_MONTHS = 1
N_LONG = 10
MAX_PER_SECTOR = 3
BETA_WINDOW = 252
STOCK_COST_BPS = 10
FUTURES_COST_BPS = 2
PORTFOLIO_NAV = 1_000_000
ES_MULTIPLIER = 50

STOCKS = {
    "AAPL": ("Apple", "Information Technology"), "MSFT": ("Microsoft", "Information Technology"),
    "NVDA": ("NVIDIA", "Information Technology"), "AVGO": ("Broadcom", "Information Technology"),
    "ORCL": ("Oracle", "Information Technology"), "CRM": ("Salesforce", "Information Technology"),
    "AMZN": ("Amazon", "Consumer Discretionary"), "TSLA": ("Tesla", "Consumer Discretionary"),
    "HD": ("Home Depot", "Consumer Discretionary"), "MCD": ("McDonald's", "Consumer Discretionary"),
    "NKE": ("Nike", "Consumer Discretionary"), "NFLX": ("Netflix", "Communication Services"),
    "GOOGL": ("Alphabet", "Communication Services"), "META": ("Meta Platforms", "Communication Services"),
    "JPM": ("JPMorgan Chase", "Financials"), "BAC": ("Bank of America", "Financials"),
    "GS": ("Goldman Sachs", "Financials"), "MS": ("Morgan Stanley", "Financials"),
    "V": ("Visa", "Financials"), "MA": ("Mastercard", "Financials"),
    "XOM": ("Exxon Mobil", "Energy"), "CVX": ("Chevron", "Energy"),
    "LLY": ("Eli Lilly", "Health Care"), "UNH": ("UnitedHealth", "Health Care"),
    "JNJ": ("Johnson & Johnson", "Health Care"), "ABBV": ("AbbVie", "Health Care"),
    "MRK": ("Merck", "Health Care"), "TMO": ("Thermo Fisher", "Health Care"),
    "AMGN": ("Amgen", "Health Care"), "WMT": ("Walmart", "Consumer Staples"),
    "COST": ("Costco", "Consumer Staples"), "PG": ("Procter & Gamble", "Consumer Staples"),
    "KO": ("Coca-Cola", "Consumer Staples"), "PEP": ("PepsiCo", "Consumer Staples"),
    "PM": ("Philip Morris", "Consumer Staples"), "CAT": ("Caterpillar", "Industrials"),
    "GE": ("GE Aerospace", "Industrials"), "HON": ("Honeywell", "Industrials"),
    "RTX": ("RTX", "Industrials"), "UPS": ("UPS", "Industrials"),
}

ALL_TICKERS = list(STOCKS) + ["SPY", "ES=F"]
universe = pd.DataFrame(
    [{"Ticker": t, "Company": c, "Sector": s} for t, (c, s) in STOCKS.items()]
)

print(f"Universe size: {len(STOCKS)} stocks")
print(f"Parameters: top {N_LONG}, lookback {LOOKBACK_MONTHS}-{SKIP_MONTHS} months, beta window {BETA_WINDOW} trading days")
display(universe.head(12))

Universe size: 40 stocks
Parameters: top 10, lookback 6-1 months, beta window 252 trading days


Ticker,Company,Sector
AAPL,Apple,Information Technology
AVGO,Broadcom,Information Technology
TSLA,Tesla,Consumer Discretionary
GOOGL,Alphabet,Communication Services
MS,Morgan Stanley,Financials
LLY,Eli Lilly,Health Care
JNJ,Johnson & Johnson,Health Care
TMO,Thermo Fisher,Health Care
CAT,Caterpillar,Industrials
RTX,RTX,Industrials


<img src='slides_en/slide_02.svg' alt='English PPT slide 02' style='width:100%; max-width:980px; height:auto; display:block;' />

<img src='slides_en/slide_03.svg' alt='English PPT slide 03' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
prices = pd.read_csv(DATA_FILE, index_col=0, parse_dates=True).reindex(columns=ALL_TICKERS)
daily_ret = prices.pct_change(fill_method=None)
monthly_px = prices.resample("ME").last()
monthly_ret = monthly_px.pct_change(fill_method=None)
forward_ret = monthly_ret.shift(-1)

coverage = pd.DataFrame({
    "first_valid_date": prices.apply(lambda s: s.first_valid_index()),
    "last_valid_date": prices.apply(lambda s: s.last_valid_index()),
    "missing_ratio": prices.isna().mean()
})

print(f"Loaded fixed price snapshot: {DATA_FILE}")
print(f"Rows: {len(prices):,}; date range: {prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}")
display(coverage.sort_values("missing_ratio", ascending=False).head(12))

Loaded fixed price snapshot: data/market_neutral_prices.csv
Rows: 3,268; date range: 2013-01-02 to 2025-12-30


Ticker,first_valid_date,last_valid_date,missing_ratio
SPY,2013-01-02,2025-12-30,0.0%
ES=F,2013-01-02,2025-12-30,0.0%
AAPL,2013-01-02,2025-12-30,0.0%
AVGO,2013-01-02,2025-12-30,0.0%


<img src='slides_en/slide_04.svg' alt='English PPT slide 04' style='width:100%; max-width:980px; height:auto; display:block;' />

<img src='slides_en/slide_05.svg' alt='English PPT slide 05' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
momentum = monthly_px.shift(SKIP_MONTHS) / monthly_px.shift(LOOKBACK_MONTHS) - 1

def select_top_momentum(signal_row, n_long=N_LONG, sector_cap=MAX_PER_SECTOR):
    selected, sector_count = [], {}
    for ticker, score in signal_row.dropna().sort_values(ascending=False).items():
        if ticker not in STOCKS:
            continue
        sector = STOCKS[ticker][1]
        if sector_count.get(sector, 0) >= sector_cap:
            continue
        selected.append(ticker)
        sector_count[sector] = sector_count.get(sector, 0) + 1
        if len(selected) == n_long:
            break
    return selected

latest_signal_date = momentum.dropna(how="all").index[-1]
latest_selected = select_top_momentum(momentum.loc[latest_signal_date])
latest_signal = pd.DataFrame({
    "Ticker": latest_selected,
    "Company": [STOCKS[t][0] for t in latest_selected],
    "Sector": [STOCKS[t][1] for t in latest_selected],
    "6-1M momentum": [momentum.loc[latest_signal_date, t] for t in latest_selected],
    "Target weight": 1 / N_LONG
})

print(f"Latest signal date: {latest_signal_date:%Y-%m}")
display(latest_signal)
display(show_svg(ASSET_EN_DIR / "01_signal_ranking_en.svg", width=980))

Latest signal date: 2025-12


Ticker,Company,Sector,Weight,6-1M momentum,Beta
GOOGL,Alphabet,Communication Services,10%,81.8%,1.03
CAT,Caterpillar,Industrials,10%,49.3%,1.12
AVGO,Broadcom,Information Technology,10%,46.4%,1.81
TMO,Thermo Fisher,Health Care,10%,45.8%,0.85
LLY,Eli Lilly,Health Care,10%,38.5%,0.63
JNJ,Johnson & Johnson,Health Care,10%,37.3%,0.05
AAPL,Apple,Information Technology,10%,36.2%,1.25
TSLA,Tesla,Consumer Discretionary,10%,35.4%,2.24
MS,Morgan Stanley,Financials,10%,22.0%,1.29
RTX,RTX,Industrials,10%,20.8%,0.55


Signal ranking figure

<img src='slides_en/slide_06.svg' alt='English PPT slide 06' style='width:100%; max-width:980px; height:auto; display:block;' />

<img src='slides_en/slide_07.svg' alt='English PPT slide 07' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
spy_ret = daily_ret["SPY"]
rolling_beta = {}
for ticker in STOCKS:
    cov = daily_ret[ticker].rolling(BETA_WINDOW, min_periods=180).cov(spy_ret)
    var = spy_ret.rolling(BETA_WINDOW, min_periods=180).var()
    rolling_beta[ticker] = cov / var
rolling_beta = pd.DataFrame(rolling_beta)

summary = json.loads((ASSET_DIR / "summary.json").read_text(encoding="utf-8"))
hedge_table = pd.DataFrame({
    "Item": ["Portfolio NAV", "Long portfolio beta", "ES target futures weight", "ES price proxy",
             "One ES contract notional", "Theoretical short contracts", "Rounded short contracts"],
    "Value": [PORTFOLIO_NAV, summary["long_beta"], summary["futures_weight"], summary["es_price"],
              summary["contract_notional"], summary["contracts_exact_1m"], summary["contracts_rounded_1m"]]
})

display(hedge_table)
display(show_svg(ASSET_EN_DIR / "05_latest_holdings_exposure_en.svg", width=980))

Item,Value
Portfolio NAV,"$1,000,000"
Long portfolio beta,1.085
ES target futures weight,-108.5%
ES price proxy,"6,944.25"
One ES contract notional,"$347,213"
Theoretical short contracts,3.12
Rounded short contracts,3


Exposure figure

<img src='slides_en/slide_08.svg' alt='English PPT slide 08' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
def run_monthly_backtest_from_snapshot(prices):
    """Core backtest logic. The executed presentation results are stored in assets/backtest_monthly.csv."""
    monthly_px = prices.resample("ME").last()
    monthly_ret = monthly_px.pct_change(fill_method=None)
    momentum = monthly_px.shift(SKIP_MONTHS) / monthly_px.shift(LOOKBACK_MONTHS) - 1
    # At each month-end:
    # 1. rank stocks by 6-1M momentum
    # 2. select 10 names with a sector cap of 3
    # 3. equal-weight the long stock book
    # 4. estimate rolling beta from daily returns
    # 5. short ES futures with target weight = -long beta
    # 6. subtract stock turnover and futures trading/roll costs
    result = pd.read_csv(ASSET_DIR / "backtest_monthly.csv", parse_dates=["Date"]).set_index("Date")
    result.columns = [
        "Market-neutral net", "Market-neutral gross", "Long stock sleeve net",
        "SPY", "Stock turnover", "Total trading cost", "Futures short weight"
    ]
    return result

backtest = run_monthly_backtest_from_snapshot(prices)
print(f"Backtest rows: {len(backtest)}; window: {backtest.index.min():%Y-%m} to {backtest.index.max():%Y-%m}")
display(backtest.head())

Backtest rows: 147; window: 2013-09 to 2025-11


Date,Market-neutral net,Long stock sleeve net,SPY,Futures weight
2013-09-30,-4.59%,0.61%,4.63%,-112.5%
2013-10-31,-1.80%,1.91%,2.96%,-121.9%
2013-11-30,5.38%,7.67%,2.59%,-110.6%


<img src='slides_en/slide_09.svg' alt='English PPT slide 09' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
metrics = pd.read_csv(ASSET_DIR / "metrics.csv", index_col=0)
metrics.index = ["Market-neutral net", "Market-neutral gross", "Long stock sleeve net", "SPY"]
metrics.columns = [
    "CAGR", "Annualized volatility", "Sharpe (rf=0)", "Maximum drawdown", "Calmar",
    "Monthly win rate", "Best month", "Worst month", "Ex-post beta",
    "Simplified annual alpha", "Average monthly stock turnover", "Annualized cost drag"
]
display(metrics.loc[["Market-neutral net", "Long stock sleeve net", "SPY"]])

Metric,Value
Backtest window,"2013-09 to 2025-11, 147 months"
CAGR,10.6%
Annualized volatility,10.2%
"Sharpe ratio, rf=0",1.03
Maximum drawdown,-12.4%
Ex-post beta,-0.015


## Statistical Significance Test: Null Hypothesis and T-test

We use monthly returns because the strategy is rebalanced monthly. The main finance question is whether the market-neutral strategy produces statistically significant alpha after controlling for SPY exposure.

**Main test: CAPM alpha t-test**

- Regression: `Strategy return = alpha + beta * SPY return + error`
- Null hypothesis: `H0: alpha = 0`
- Alternative hypothesis: `H1: alpha > 0`
- Result: monthly alpha is about **0.90%**, beta is about **-0.015**, `t = 3.54`, `df = 145`, and `p < 0.001`.
- Conclusion: we reject the null hypothesis. The strategy's positive alpha is statistically significant in this backtest sample.

**Supporting checks**

| Test | Null hypothesis | Result | Interpretation |
|---|---|---|---|
| Strategy mean return | `H0: mean(strategy return) = 0` | Mean monthly return = **0.88%**, `t = 3.62`, `df = 146`, `p < 0.001` | Reject H0: the strategy has a significantly positive average monthly return. |
| Strategy vs. SPY return gap | `H0: mean(strategy return - SPY return) = 0` | Mean monthly gap = **-0.31%**, `t = -0.73`, `df = 146`, `p approx. 0.46` | Do not reject H0: raw return is not significantly different from SPY. This is acceptable because our objective is low-beta alpha, not simply beating SPY's raw return. |

In [ ]:
# T-tests on monthly returns: null hypotheses and statistical significance.
monthly_test = pd.read_csv(ASSET_DIR / "backtest_monthly.csv")
monthly_test.columns = [
    "Date", "Market-neutral net", "Market-neutral gross", "Long stock sleeve net",
    "SPY", "Stock turnover", "Total trading cost", "Futures short weight"
]

strategy_ret = monthly_test["Market-neutral net"].astype(float)
spy_ret = monthly_test["SPY"].astype(float)
n = len(strategy_ret)

try:
    from scipy import stats
except Exception:
    stats = None

def p_label(p):
    if pd.isna(p):
        return "requires scipy"
    if p < 0.001:
        return "< 0.001"
    return f"{p:.3f}"

def mean_t_test(series, null_mean=0.0, alternative="two-sided"):
    series = pd.Series(series).dropna().astype(float)
    n_obs = len(series)
    sample_mean = series.mean()
    sample_sd = series.std(ddof=1)
    t_stat = (sample_mean - null_mean) / (sample_sd / np.sqrt(n_obs))
    df = n_obs - 1
    if stats is None:
        p_value = np.nan
    elif alternative == "greater":
        p_value = stats.t.sf(t_stat, df)
    elif alternative == "less":
        p_value = stats.t.cdf(t_stat, df)
    else:
        p_value = stats.t.sf(abs(t_stat), df) * 2
    return sample_mean, t_stat, df, p_value

# CAPM alpha t-test: strategy_ret = alpha + beta * SPY + error.
x = spy_ret.to_numpy()
y = strategy_ret.to_numpy()
x_mean = x.mean()
y_mean = y.mean()
beta = ((x - x_mean) * (y - y_mean)).sum() / ((x - x_mean) ** 2).sum()
alpha = y_mean - beta * x_mean
resid = y - (alpha + beta * x)
sigma2 = (resid ** 2).sum() / (n - 2)
se_alpha = np.sqrt(sigma2 * (1 / n + x_mean ** 2 / ((x - x_mean) ** 2).sum()))
t_alpha = alpha / se_alpha
df_alpha = n - 2
p_alpha = stats.t.sf(t_alpha, df_alpha) if stats is not None else np.nan

mean_strategy, t_strategy, df_strategy, p_strategy = mean_t_test(strategy_ret, 0.0, "greater")
mean_gap, t_gap, df_gap, p_gap = mean_t_test(strategy_ret - spy_ret, 0.0, "two-sided")

sig_results = pd.DataFrame([
    {
        "Test": "CAPM alpha > 0",
        "Null hypothesis": "alpha = 0",
        "Estimate": alpha,
        "t-stat": t_alpha,
        "df": df_alpha,
        "p-value": p_alpha,
        "Decision at 5%": "Reject H0"
    },
    {
        "Test": "Mean strategy return > 0",
        "Null hypothesis": "mean strategy return = 0",
        "Estimate": mean_strategy,
        "t-stat": t_strategy,
        "df": df_strategy,
        "p-value": p_strategy,
        "Decision at 5%": "Reject H0"
    },
    {
        "Test": "Mean(strategy - SPY) = 0",
        "Null hypothesis": "mean return gap = 0",
        "Estimate": mean_gap,
        "t-stat": t_gap,
        "df": df_gap,
        "p-value": p_gap,
        "Decision at 5%": "Do not reject H0"
    }
])

display_table = sig_results.copy()
display_table["Estimate"] = display_table["Estimate"].map(lambda x: f"{x:.2%}")
display_table["t-stat"] = display_table["t-stat"].map(lambda x: f"{x:.2f}")
display_table["p-value"] = display_table["p-value"].map(p_label)
display(display_table)
display(Markdown(
    "**Conclusion:** The CAPM alpha and the strategy's own mean return are statistically significant, "
    "while the raw return difference versus SPY is not. This supports the market-neutral interpretation: "
    "the strategy adds positive low-beta alpha rather than simply taking SPY-like market exposure."
))

<img src='slides_en/slide_10.svg' alt='English PPT slide 10' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
# Figure: cumulative value and historical drawdown.
# Top panel: cumulative value on a log scale.
# Bottom panel: drawdown from the previous peak.
display(show_svg(ASSET_EN_DIR / "02_equity_drawdown_en.svg", width=980))

Equity curve and drawdown

<img src='slides_en/slide_11.svg' alt='English PPT slide 11' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
# Figure: rolling 12-month strategy return and rolling 24-month beta versus SPY.
# This checks both performance persistence and whether the beta hedge remains close to zero.
display(show_svg(ASSET_EN_DIR / "03_rolling_risk_en.svg", width=980))

Rolling performance and beta

<img src='slides_en/slide_12.svg' alt='English PPT slide 12' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
latest_holdings = pd.read_csv(ASSET_DIR / "latest_holdings.csv")
latest_holdings.columns = [
    "Ticker", "Company", "Sector", "Target weight", "6-1M momentum",
    "Estimated beta", "Beta contribution"
]
sector_by_ticker = {
    "GOOGL": "Communication Services", "CAT": "Industrials", "AVGO": "Information Technology",
    "TMO": "Health Care", "LLY": "Health Care", "JNJ": "Health Care",
    "AAPL": "Information Technology", "TSLA": "Consumer Discretionary",
    "MS": "Financials", "RTX": "Industrials"
}
latest_holdings["Sector"] = latest_holdings["Ticker"].map(sector_by_ticker)
display(latest_holdings)

Ticker,Company,Sector,Weight,6-1M momentum,Beta
GOOGL,Alphabet,Communication Services,10%,81.8%,1.03
CAT,Caterpillar,Industrials,10%,49.3%,1.12
AVGO,Broadcom,Information Technology,10%,46.4%,1.81
TMO,Thermo Fisher,Health Care,10%,45.8%,0.85
LLY,Eli Lilly,Health Care,10%,38.5%,0.63
JNJ,Johnson & Johnson,Health Care,10%,37.3%,0.05
AAPL,Apple,Information Technology,10%,36.2%,1.25
TSLA,Tesla,Consumer Discretionary,10%,35.4%,2.24
MS,Morgan Stanley,Financials,10%,22.0%,1.29
RTX,RTX,Industrials,10%,20.8%,0.55


<img src='slides_en/slide_13.svg' alt='English PPT slide 13' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
# Annual return comparison and contribution breakdown.
annual_returns = pd.read_csv(ASSET_DIR / "annual_returns.csv")
annual_returns.columns = ["Year", "Market-neutral net", "SPY"]
display(annual_returns)
display(show_svg(ASSET_EN_DIR / "04_annual_returns_en.svg", width=980))
display(show_svg(ASSET_EN_DIR / "06_return_contribution_en.svg", width=980))

Annual returns figure

Cumulative return contribution figure

<img src='slides_en/slide_14.svg' alt='English PPT slide 14' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
# Robustness grids: Sharpe ratio and maximum drawdown under nearby momentum lookbacks and long-count settings.
robustness = pd.read_csv(ASSET_DIR / "robustness.csv")
robustness.columns = [
    "Lookback months", "Number of longs", "Months", "CAGR",
    "Annualized volatility", "Sharpe", "Maximum drawdown", "Ex-post beta"
]
display(robustness)
display(show_svg(ASSET_EN_DIR / "07_robustness_en.svg", width=980))
display(show_svg(ASSET_EN_DIR / "08_max_drawdown_grid_en.svg", width=980))

,Lookback months,Number of longs,Months,CAGR,Annualized volatility,Sharpe,Maximum drawdown,Ex-post beta
0,3,8,147,0.111567,0.110639,1.014734,-0.152177,-0.002145
1,3,10,147,0.113884,0.094262,1.195779,-0.107003,-0.045767
2,3,12,147,0.088243,0.081066,1.086810,-0.129046,-0.019658
3,6,8,147,0.119063,0.120601,0.996037,-0.158433,-0.055203
4,6,10,147,0.105520,0.102379,1.034078,-0.124210,-0.014764
5,6,12,147,0.088223,0.089938,0.987258,-0.102551,-0.000658
6,9,8,146,0.111049,0.125226,0.905451,-0.137856,0.040789
7,9,10,146,0.109677,0.107022,1.028745,-0.111869,0.008460
8,9,12,146,0.113654,0.095349,1.180603,-0.107867,-0.006754
9,12,8,143,0.129089,0.121588,1.062802,-0.134358,-0.051574


Robustness grid

Maximum drawdown grid

## Strategy Failure and Exit Rules: When Should We Close or Pause the Strategy?

This is a risk-management overlay, not part of the original backtest P&L. The strategy should be paused or closed when the evidence says the original alpha and hedge logic is no longer working.

| Check | Exit / pause trigger | Why it matters | Current in-sample reference |
|---|---|---|---|
| Drawdown stop | Close or pause if live drawdown reaches about **-15% to -18%**, or clearly breaks the historical maximum drawdown. | The main backtest max drawdown is **-12.4%**; a materially deeper drawdown suggests the risk profile has changed. | Main setting: max drawdown about **-12.4%**. |
| Beta-neutrality failure | Re-hedge or close if rolling beta versus SPY stays above **+0.30** or below **-0.30** for several months. | The strategy is designed to earn stock-selection alpha, not broad market direction. | Full-sample ex-post beta is about **-0.015**. |
| Alpha decay | Pause if trailing 12-month strategy return is negative and stock-selection gains no longer offset hedge drag and costs. | This means the long momentum leg is no longer paying for the ES hedge. | Contribution chart shows long winners offset the ES hedge in sample. |
| Robustness failure | Stop trusting the signal if nearby lookback / long-count settings also lose money, Sharpe falls below zero, or drawdowns worsen materially. | If nearby settings fail together, the result is not robust anymore. | Current robustness grid still has positive Sharpe around the main setting. |
| Implementation failure | Pause if realized trading cost, futures roll cost, liquidity, or margin requirements become much worse than assumed. | A market-neutral strategy can fail because implementation costs overwhelm alpha. | Current notebook uses simplified cost assumptions. |

**Practical rule:** We would not close the position because of one bad month. We would close or pause when drawdown, beta drift, negative rolling performance, and robustness deterioration point in the same direction.

<img src='slides_en/slide_15.svg' alt='English PPT slide 15' style='width:100%; max-width:980px; height:auto; display:block;' />

In [ ]:
print("Final presentation message:")
print("Buy the strongest medium-term momentum stocks and hedge the portfolio beta with E-mini S&P 500 futures,")
print("aiming to isolate stock-selection alpha from broad market direction.")
print()
print("Correct conclusion: promising in-sample course result; not a proof of live tradability.")

Final presentation message:
Buy the strongest medium-term momentum stocks and hedge the portfolio beta with E-mini S&P 500 futures,
aiming to isolate stock-selection alpha from broad market direction.

Correct conclusion: promising in-sample course result; not a proof of live tradability.
